Import thư viện và load dữ liệu



In [1]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

def load_file(filename):
    path = Path(filename)
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {filename}")
    return joblib.load(path)

X_train_balance = load_file("X_train_balance.pkl")
X_test_balance = load_file("X_test_balance.pkl")
y_train_balance = np.asarray(load_file("y_train_balance.pkl")).astype(int)
y_test_balance = np.asarray(load_file("y_test_balance.pkl")).astype(int)

print("X_train_balance:", X_train_balance.shape)
print("X_test_balance :", X_test_balance.shape)
print("y_train_balance:", y_train_balance.shape)
print("y_test_balance :", y_test_balance.shape)

def show_label_distribution(y, name):
    labels, counts = np.unique(y, return_counts=True)
    print(f"\n{name}")
    for label, count in zip(labels, counts):
        label_name = "Spam" if label == 1 else "Ham"
        print(f"{label} ({label_name}): {count} mẫu - {count / len(y) * 100:.2f}%")

show_label_distribution(y_train_balance, "Train label distribution")
show_label_distribution(y_test_balance, "Test label distribution")

X_train_balance: (6552, 10000)
X_test_balance : (1639, 10000)
y_train_balance: (6552,)
y_test_balance : (1639,)

Train label distribution
0 (Ham): 3303 mẫu - 50.41%
1 (Spam): 3249 mẫu - 49.59%

Test label distribution
0 (Ham): 826 mẫu - 50.40%
1 (Spam): 813 mẫu - 49.60%


Class Multinomial Naive Bayes 

Naive Bayes học bằng xác suất:

- `P(Ham)`, `P(Spam)` gọi là prior probability
- `P(feature | Ham)`, `P(feature | Spam)` gọi là conditional probability
- Dự đoán class có posterior probability cao hơn

`alpha` là tham số smoothing, giúp tránh xác suất bằng 0.


In [2]:
class MultinomialNaiveBayesFromScratch:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes_ = None
        self.class_log_prior_ = None
        self.feature_log_prob_ = None

    def fit(self, X, y):
        y = np.asarray(y).astype(int)
        self.classes_ = np.unique(y)

        n_samples, n_features = X.shape
        self.class_log_prior_ = np.zeros(len(self.classes_))
        self.feature_log_prob_ = np.zeros((len(self.classes_), n_features))

        for i, c in enumerate(self.classes_):
            X_c = X[y == c]

            # Prior probability: log P(class)
            self.class_log_prior_[i] = np.log(X_c.shape[0] / n_samples)

            # Conditional probability: log P(feature | class)
            feature_count = np.asarray(X_c.sum(axis=0)).ravel()
            smoothed_count = feature_count + self.alpha
            self.feature_log_prob_[i] = np.log(smoothed_count / smoothed_count.sum())

        return self

    def predict_log_proba(self, X):
        joint = X.dot(self.feature_log_prob_.T) + self.class_log_prior_
        joint = np.asarray(joint)

        # log-sum-exp để chuẩn hóa xác suất
        max_log = np.max(joint, axis=1, keepdims=True)
        log_sum = max_log + np.log(np.sum(np.exp(joint - max_log), axis=1, keepdims=True))
        return joint - log_sum

    def predict_proba(self, X):
        return np.exp(self.predict_log_proba(X))

    def predict(self, X):
        log_proba = self.predict_log_proba(X)
        return self.classes_[np.argmax(log_proba, axis=1)]

 Hàm đánh giá 

In [3]:
def confusion_values(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))

    return TN, FP, FN, TP

def evaluate(y_true, y_pred):
    TN, FP, FN, TP = confusion_values(y_true, y_pred)

    total = TP + TN + FP + FN
    accuracy = (TP + TN) / total if total else 0
    precision = TP / (TP + FP) if (TP + FP) else 0
    recall = TP / (TP + FN) if (TP + FN) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "TN": TN,
        "FP": FP,
        "FN": FN,
        "TP": TP
    }

def log_loss_score(y_true, y_proba, classes):
    class_to_index = {c: i for i, c in enumerate(classes)}
    true_index = np.array([class_to_index[y] for y in y_true])

    p = y_proba[np.arange(len(y_true)), true_index]
    p = np.clip(p, 1e-15, 1 - 1e-15)

    return float(-np.mean(np.log(p)))

Train mô hình với giá trị alpha

In [4]:
final_alpha = 0.1

final_model = MultinomialNaiveBayesFromScratch(alpha=final_alpha)
final_model.fit(X_train_balance, y_train_balance)

y_train_balance_pred = final_model.predict(X_train_balance)
y_test_balance_pred = final_model.predict(X_test_balance)

y_train_balance_proba = final_model.predict_proba(X_train_balance)
y_test_balance_proba = final_model.predict_proba(X_test_balance)

print("Đã train và predict với alpha =", final_alpha)

Đã train và predict với alpha = 0.1


Đánh giá mô hình cuối cùng

In [5]:
test_metrics = evaluate(y_test_balance, y_test_balance_pred)

test_loss = log_loss_score(y_test_balance, y_test_balance_proba, final_model.classes_)

result_table = pd.DataFrame({
    "Metric": [
        "Test Accuracy",
        "Test Precision",
        "Test Recall",
        "Test F1-score",
        "Test Log Loss",
        "TN", "FP", "FN", "TP"
    ],
    "Value": [
        f"{test_metrics['Accuracy'] * 100:.2f}%",
        f"{test_metrics['Precision'] * 100:.2f}%",
        f"{test_metrics['Recall'] * 100:.2f}%",
        f"{test_metrics['F1-score'] * 100:.2f}%",
        f"{test_loss * 100:.2f}%",
        int(test_metrics["TN"]),
        int(test_metrics["FP"]),
        int(test_metrics["FN"]),
        int(test_metrics["TP"])
    ]
})

result_table

,Metric,Value
0,Test Accuracy,98.66%
1,Test Precision,99.38%
2,Test Recall,97.91%
3,Test F1-score,98.64%
4,Test Log Loss,3.14%
5,TN,821
6,FP,5
7,FN,17
8,TP,796


Confusion Matrix

In [6]:
confusion_matrix_table = pd.DataFrame(
    [
        [test_metrics["TN"], test_metrics["FP"]],
        [test_metrics["FN"], test_metrics["TP"]],
    ],
    index=["Actual Ham (0)", "Actual Spam (1)"],
    columns=["Predicted Ham (0)", "Predicted Spam (1)"]
)

confusion_matrix_table

,Predicted Ham (0),Predicted Spam (1)
Actual Ham (0),821,5
Actual Spam (1),17,796
